<img src=images/cards.png align=right width=170px>

# Hackathons

There are two hackathons in this notebook:

1. Make a card game with a Deck of cards
2. Make a bank account

## 1. Make a card game with a Deck of cards

In this hackathon you will be extending the already existing deck of cards below to create a game.

The game goes like this:
- 2+ players in a game
- Each player gets one full deck of cards, shuffled
- Each player deals 15 cards and discards them
- With the remaining cards, the players add up the total number of face cards (J, Q, K)
- The player with the most face cards wins
- If there is a draw, the player with the most Ks wins (then most Qs, Js etc.)

The goal of this challenge is to use the `__lt__`, `__gt__`, `__eq__`, `__ge__`, `__le__` dunder methods to compare the decks with the remaining cards.

In [2]:
import collections

Card = collections.namedtuple("Card", ["rank", "suit"])


class Deck:
    ranks = "23456789TJQKA"
    suits = "♠♥♦♣"

    def __init__(self, debug_value=False):
        self._original_cards = [
            Card(rank, suit) for suit in self.suits for rank in self.ranks
        ]

    def __len__(self):
        return len(self._original_cards)

    def __str__(self):
        return f"Deck(suits={self.suits}, ranks={self.ranks})"

    def __getitem__(self, position):
        return self._original_cards[position]

    def __setitem__(self, ind, value):
        self._original_cards[ind] = value

    def deal(self):
        return self._original_cards.pop()

    def check_ace(self):
        return self._original_cards[-1].rank == "A"

    @property
    def cards(self):
        return self._original_cards

    def _score(self):
        ranks = [card.rank for card in self._original_cards]
        return (
            sum(rank in {"J", "Q", "K"} for rank in ranks),
            sum(rank == "K" for rank in ranks),
            sum(rank in "Q" for rank in ranks),
            sum(rank in "J" for rank in ranks),
        )


deck = Deck()

In [6]:
from random import shuffle

player1 = Deck()
player2 = Deck()
shuffle(player1)  # shuffle the player's deck
shuffle(player2)  # shuffle the player's deck

for _ in range(15):
    player1.deal()  # deal 15 cards to player 1
    player2.deal()  # deal 15 cards to player 2

if player1._score()[0] > player2._score()[0]:
    print("Player 1 wins")
elif player2._score()[0] > player1._score()[0]:
    print("Player 2 wins")
elif player1._score()[1] > player2._score()[1]:
    print("Player 1 wins")
elif player2._score()[1] > player1._score()[1]:
    print("Player 2 wins")
elif player1._score()[2] > player2._score()[2]:
    print("Player 1 wins")
elif player2._score()[2] > player1._score()[2]:
    print("Player 2 wins")
elif player1._score()[3] > player2._score()[3]:
    print("Player 1 wins")
elif player2._score()[3] > player1._score()[3]:
    print("Player 2 wins")
else:
    print("Tie")

Player 1 wins


In [ ]:
# %load answers/ex-bonus-3-compare.py
import collections
from random import shuffle


class Deck:
    def __init__(self, name):
        self.name = name

        Card = collections.namedtuple("Card", ["rank", "suit"])
        self._cards = [Card(rank, suit) for suit in self.suits for rank in self.ranks]
        self.dealt_cards = []

    def __len__(self):
        return len(self._cards)

    def __str__(self):
        return f"Deck(suits={self.suits}, ranks={self.ranks})"

    def __getitem__(self, position):
        return self._cards[position]

    def __setitem__(self, ind, val):
        self._cards[ind] = val

    def __add__(self, other):
        return self._cards + other._cards

    def __gt__(self, other):
        if self.__class__ == other.__class__:
            return self.num_j_q_k > other.num_j_q_k

        return NotImplemented

    def __lt__(self, other):
        if self.__class__ == other.__class__:
            return self.num_j_q_k < other.num_j_q_k

        return NotImplemented

    def __eq__(self, other):
        if self.__class__ == other.__class__:
            return self.num_j_q_k == other.num_j_q_k

        return NotImplemented

    @property
    def num_j(self):
        return len([card for card in self._cards if card.rank == "J"])

    @property
    def num_q(self):
        return len([card for card in self._cards if card.rank == "Q"])

    @property
    def num_k(self):
        return len([card for card in self._cards if card.rank == "K"])

    @property
    def num_j_q_k(self):
        return self.num_j + self.num_q + self.num_k

    def win(self, other):
        """Determine winner and print result. Returns winner name or None for draw."""
        # Primary comparison: total face cards
        if self > other:
            print(f"{self.name} wins with {self.num_j_q_k} face cards (J, Q, K)")
            return self.name
        elif self < other:
            print(f"{other.name} wins with {other.num_j_q_k} face cards (J, Q, K)")
            return other.name

        # Tiebreaker: check K, Q, J in order
        for rank, count_attr in [("K", "num_k"), ("Q", "num_q"), ("J", "num_j")]:
            self_count = getattr(self, count_attr)
            other_count = getattr(other, count_attr)

            if self_count > other_count:
                print(f"{self.name} wins with {self_count} {rank}s")
                return self.name
            elif self_count < other_count:
                print(f"{other.name} wins with {other_count} {rank}s")
                return other.name

        # All counts equal - draw
        print(f"{self.name} reaches a draw with {other.name}")
        return None

    def deal(self):
        return self._cards.pop()


class French52Deck(Deck):
    ranks = "23456789TJQKA"
    suits = "♠♥♦♣"

    def top_card_is_ace(self):
        return self.cards[-1].rank == "A"


deck1 = French52Deck("Tom")
deck2 = French52Deck("Jerry")

print(f"Shuffling the cards for {deck1.name} and {deck2.name}...")
shuffle(deck1)
shuffle(deck2)

for i in range(15):
    deck1.deal()
    deck2.deal()

print(
    f"After discarding 15 cards, there are {len(deck1)} cards for {deck1.name} and {len(deck2)} cards for {deck2.name}."
)

print(
    f"{deck1.name} has {deck1.num_j_q_k} face cards (J, Q, K) with {deck1.num_j} Js, {deck1.num_q} Qs, {deck1.num_k} Ks."
)
print(
    f"{deck2.name} has {deck2.num_j_q_k} face cards (J, Q, K) with {deck2.num_j} Js, {deck2.num_q} Qs, {deck2.num_k} Ks."
)

winner = deck1.win(deck2)
print(f"The winner is {winner}")

## 2. Make a bank account

Using what you've learned about object oriented programming, make a class that works as a bank account.

Your banking account should have the following properties (and any other functionality you can think of):
- When opened (initialized) it should have a starting balance of 0 
- The owner should be able to 
    - deposit money (add money to the balance)
    - withdraw money, as long as the withdraw amount doesn't put them over their overdraft limit
    - increase their overdraft limit (starts at 0) by 100 each time

In [ ]:
class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance